In [ ]:
# ============================================================
# 02 — BM25 RETRIEVAL (MIND)
# Lexical retrieval from full pool -> recall@K {50,100,200}
# Fully self-contained MIND notebook. Hardcoded paths.
# ============================================================
!pip install lightgbm sentence-transformers -q
import os, glob, re, math, time, zipfile, numpy as np, pandas as pd, datetime as dt, lightgbm as lgb, random, warnings
warnings.filterwarnings("ignore")
from bisect import bisect_left
from collections import defaultdict, Counter
random.seed(0)
# ---- hardcoded MIND paths (small: train -> dev for offline metrics) ----
TRAIN = "/kaggle/input/datasets/arashnic/mind-news-dataset/MINDsmall_train"
DEV   = "/kaggle/input/datasets/wrathofgod123/mind-dev/MINDsmall_dev"
SPLITS = [TRAIN, DEV]
NEWS = ["news_id","category","subcategory","title","abstract","url","te","ae"]
BEH  = ["impression_id","user_id","time","history","impressions"]
_WORD = re.compile(r"[^\W\d_]+", re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if isinstance(t,str) else []
def pfx(x): return f"mind:{x}"

news = pd.concat([pd.read_csv(f"{d}/news.tsv", sep="\t", header=None, names=NEWS, quoting=3,
                 usecols=["news_id","category","title","abstract"]) for d in SPLITS]
                ).drop_duplicates("news_id").reset_index(drop=True)
news["title"] = news["title"].fillna(""); news["abstract"] = news["abstract"].fillna("")
cat_lut = {pfx(r.news_id):(r.category if isinstance(r.category,str) else "") for r in news.itertuples()}
ids = [pfx(r.news_id) for r in news.itertuples()]
corpus = [tok(f"{r.title} {r.abstract}") for r in news.itertuples()]
id_to_row = {x:i for i,x in enumerate(ids)}
title_lut = {pfx(r.news_id):tok(r.title) for r in news.itertuples()}
print("articles:", len(ids))


In [ ]:
class BM25:
    def __init__(s,c,k1=1.5,b=0.75):
        s.k1,s.b=k1,b;s.N=len(c);s.tf=[Counter(d) for d in c]
        s.dl=np.array([len(d) for d in c],float);s.avg=s.dl.mean()
        df=Counter()
        for t in s.tf: df.update(t.keys())
        s.idf={w:math.log((s.N-d+.5)/(d+.5)+1) for w,d in df.items()}
    def score(s,q,r):
        if not q: return 0.0
        tf=s.tf[r];dn=s.k1*(1-s.b+s.b*s.dl[r]/s.avg);v=0.0
        for w in set(q):
            f=tf.get(w,0)
            if f: v+=s.idf.get(w,0)*(f*(s.k1+1))/(f+dn)
        return v
    def scores_all(s,q):
        return np.array([s.score(q,r) for r in range(s.N)])
bm25 = BM25(corpus); print("BM25 built")


In [ ]:
def load_beh(p):
    b=pd.read_csv(f"{p}/behaviors.tsv",sep="\t",header=None,names=BEH,quoting=3)
    b["t"]=pd.to_datetime(b["time"],format="%m/%d/%Y %I:%M:%S %p",errors="coerce"); return b
b_dv=load_beh(DEV)
hist_lut={}
for b in [load_beh(TRAIN), b_dv]:
    for u,h in zip(b["user_id"],b["history"]):
        if isinstance(h,str) and h: hist_lut[pfx(u)]=[pfx(x) for x in h.split()]
def hist_q(uid,mh=30):
    ai=hist_lut.get(uid,[])[-mh:];q=[]
    for x in ai: q.extend(title_lut.get(x,[]))
    return q
print("history built; users:", len(hist_lut))


In [ ]:
# ---- recall@K: is the clicked article in the top-K retrieved from the full pool? ----
import random as _r; _r.seed(0)
eval_rows=[]
for u,imps in zip(b_dv["user_id"], b_dv["impressions"]):
    if not isinstance(imps,str): continue
    clicked=[pfx(tk.split("-")[0]) for tk in imps.split() if tk.endswith("-1")]
    uid=pfx(u)
    if clicked and hist_lut.get(uid): eval_rows.append((uid,clicked[0]))
_r.shuffle(eval_rows); eval_rows=eval_rows[:2000]
print("eval impressions:", len(eval_rows))

Ks=[50,100,200]; hits={k:0 for k in Ks}; n=0
for uid,clicked in eval_rows:
    q=hist_q(uid)
    scores=bm25.scores_all(q)
    order=np.argsort(-scores)
    crow=id_to_row.get(clicked)
    if crow is None: continue
    pos=np.where(order==crow)[0]
    if len(pos)==0: continue
    for k in Ks:
        if pos[0]<k: hits[k]+=1
    n+=1
print("\n=== MIND BM25 retrieval recall@K ===")
for k in Ks: print(f"  recall@{k}: {hits[k]/n:.4f}")
print("(lexical baseline; compare with 03 semantic)")
